In [24]:
!pip install langchain langchain-openai langsmith

import os
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [25]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "lsv2_xxx" # Replace with your actual Langsmith API Key
os.environ["LANGCHAIN_PROJECT"] = "Resume-Screening-System"

In [26]:
os.environ["GROQ_API_KEY"] = "gsk_xxxx"

In [27]:
!pip install langchain_groq

In [28]:
strong_resume = """
Data Scientist with 5 years experience.
Skills: Python, Machine Learning, Deep Learning, NLP
Tools: TensorFlow, PyTorch, Scikit-learn
"""

average_resume = """
Data Analyst with 2 years experience.
Skills: Python, SQL, Excel
Tools: Power BI
"""

weak_resume = """
Fresher graduate.
Skills: Basic C programming
Tools: None
"""

job_description = """
Looking for a Data Scientist.
Required Skills: Python, Machine Learning, NLP
Tools: TensorFlow, PyTorch
Experience: 3+ years
"""

In [29]:
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract:
- Skills
- Tools
- Experience

Rules:
- Do NOT assume anything
- Return JSON

Resume:
{resume}
"""
)

In [30]:
score_prompt = PromptTemplate(
    input_variables=["match"],
    template="""
Give score (0-100)

Criteria:
- Skills: 50%
- Experience: 30%
- Tools: 20%

Return JSON:
{{
  "score": number,
  "reason": explanation
}}

Data:
{match}
"""
)

In [31]:
explain_prompt = PromptTemplate(
    input_variables=["score"],
    template="""
Explain candidate evaluation.

Return:
- Strengths
- Weaknesses
- Final Verdict

Data:
{score}
"""
)

In [33]:
from langchain_groq import ChatGroq

def get_llm(temp=0.0):
    return ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=temp
    )

llm = get_llm() # Changed to use ChatGroq via get_llm()

extract_chain = extract_prompt | llm
match_chain = match_prompt | llm
score_chain = score_prompt | llm
explain_chain = explain_prompt | llm

In [36]:
def run_pipeline(resume):
    extracted = extract_chain.invoke({"resume": resume})

    matched = match_chain.invoke({
        "resume_data": extracted.content,
        "job_description": job_description
    })

    scored = score_chain.invoke({
        "match": matched.content
    })

    explained = explain_chain.invoke({
        "score": scored.content
    })

    return explained.content

In [37]:
print("STRONG CANDIDATE:\n")
print(run_pipeline(strong_resume))

print("\n----------------------\n")

print("AVERAGE CANDIDATE:\n")
print(run_pipeline(average_resume))

print("\n----------------------\n")

print("WEAK CANDIDATE:\n")
print(run_pipeline(weak_resume))

STRONG CANDIDATE:

**Candidate Evaluation**

### Strengths
1. **Matching Skills**: The candidate's resume matches all the required skills for the job, including Python, Machine Learning, and NLP.
2. **Exceeding Experience Requirement**: The candidate has 5 years of experience, which meets and exceeds the job requirement of 3+ years.
3. **Required Tools**: The candidate's resume shows proficiency in both required tools, TensorFlow and PyTorch.

### Weaknesses
None identified in the given data, as the candidate meets all the requirements and exceeds the experience requirement.

### Final Verdict
Based on the evaluation, the candidate has a perfect score of 100. The candidate's resume matches all the required skills, meets and exceeds the experience requirement, and matches all the required tools. This suggests that the candidate is highly qualified for the job and would be an excellent fit. 

```json
{
  "score": 100,
  "reason": "The resume matches all required skills, meets and exceeds

In [38]:
bad_resume = "I know everything in AI"

print(run_pipeline(bad_resume))

**Candidate Evaluation**

### Strengths
- The candidate has a broad mention of "AI" in their resume, indicating some level of familiarity with artificial intelligence, which could be a foundation for further learning and development in specific areas like Machine Learning and NLP.

### Weaknesses
- The candidate lacks direct experience and skills in the required areas of Machine Learning, NLP, and specific tools like TensorFlow and PyTorch.
- There is no listed experience that matches the required 3+ years, which is a significant gap for the position.
- The candidate does not demonstrate proficiency in Python, a crucial skill for the role.

### Final Verdict
Based on the evaluation criteria, the candidate scores 10 out of a possible 100. This low score is due to the minimal alignment with the required skills, the lack of direct experience, and the absence of proficiency in the specified tools. While the candidate shows a very basic level of relevance through the mention of "AI," this i